In [13]:
from embedder import Embedder

# Initialize the lightweight ONNX embedder
embedder = Embedder()

# Embed the target query
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

# Print out the first element of the vector
print(f"Vector length: {len(v)}")
print(f"First value (v[0]): {v[0]:.2f}")

Vector length: 384
First value (v[0]): -0.02


In [14]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
import numpy as np

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

doc_vector = embedder.encode(doc["content"])

similarity = np.dot(query_vector, doc_vector)

print(similarity)

0.36107027225589694


In [20]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

X = np.array(embedder.encode_batch([chunk["content"] for chunk in chunks]))

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

scores = X.dot(v)

best_idx = np.argmax(scores)

print(best_idx)
print(scores[best_idx])
print(chunks[best_idx]["filename"])

94
0.6489017718578813
02-vector-search/lessons/07-sqlitesearch-vector.md


In [25]:
from minsearch import VectorSearch

X = np.array(embedder.encode_batch([c["content"] for c in chunks]))

index = VectorSearch()
index.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
q = embedder.encode(query)

results = index.search(q, num_results=5)

for r in results:
    print(r["filename"])

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


In [26]:
from minsearch import Index, VectorSearch

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

vector_index = VectorSearch()
vector_index.fit(X, chunks)

query = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(query)[:5]

query_vector = embedder.encode(query)
vector_results = vector_index.search(query_vector, num_results=5)

print("TEXT SEARCH")
for r in text_results:
    print(r["filename"])

print("\nVECTOR SEARCH")
for r in vector_results:
    print(r["filename"])

text_files = {r["filename"] for r in text_results}
vector_files = {r["filename"] for r in vector_results}

print("\nOnly in vector search:")
print(vector_files - text_files)

TEXT SEARCH
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

VECTOR SEARCH
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md

Only in vector search:
{'02-vector-search/lessons/08-pgvector.md'}


In [29]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


query = "How do I give the model access to tools?"

text_results = text_index.search(query)[:5]

query_vector = embedder.encode(query)
vector_results = vector_index.search(query_vector, num_results=5)

results = rrf([vector_results, text_results])

for i, r in enumerate(results, start=1):
    print(f"{i}. {r['filename']}")

1. 01-agentic-rag/lessons/13-function-calling.md
2. 01-agentic-rag/lessons/01-intro.md
3. 01-agentic-rag/lessons/14-agentic-loop.md
4. 04-evaluation/lessons/02-ground-truth.md
5. 01-agentic-rag/lessons/16-other-frameworks.md
